In [13]:
# Cell 1: Setup and Libraries (FIXED)

# Install the Natural Language Toolkit library
!pip install nltk

# Import necessary libraries
import nltk
import random
import re
from nltk.stem import PorterStemmer

# Download NLTK data required for tokenization, including the missing resource
# The main 'punkt' is usually enough, but we explicitly download the missing part to fix the LookupError.
nltk.download('punkt')
nltk.download('punkt_tab') # <<< ADD THIS LINE TO FIX THE ERROR

# Initialize the stemmer
stemmer = PorterStemmer()

print("Setup complete. Libraries imported and NLTK data downloaded.")

Setup complete. Libraries imported and NLTK data downloaded.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [9]:
# Knowledge Base: Dictionary containing intents, keywords, and responses

intents = {
    # Intent 1: Greeting
    'greeting': {
        'keywords': ['hello', 'hi', 'hey', 'greetings', 'sup'],
        'responses': ["Hello! I'm the Simple Bot. How can I assist you?", "Hi there! What can I do for you today?", "Hey! Nice to chat with you."],
        'score_threshold': 1
    },

    # Intent 2: Get Business Hours
    'get_hours': {
        'keywords': ['open', 'hour', 'time', 'close', 'schedule', 'when', 'visit'],
        'responses': ["We are open Monday to Friday, from 9 AM to 5 PM.", "Our business hours are 9-5 on weekdays. Come visit us!", "You can find us here from 9am until 5pm."],
        'score_threshold': 1
    },

    # Intent 3: Ask the Bot's Name (Small Talk)
    'bot_name': {
        'keywords': ['name', 'who', 'are', 'you', 'call'],
        'responses': ["I'm the Simple Bot, built to help with basic inquiries.", "You can call me Simple Bot.", "I don't have a real name, but I can help you with your questions."],
        'score_threshold': 2 # Requires more keywords to match this specific intent
    },

    # Intent 4: Thank You/Appreciation
    'thanks': {
        'keywords': ['thank', 'thanks', 'cool', 'awesome', 'appreciate', 'helpful'],
        'responses': ["You're very welcome!", "Glad I could be helpful!", "Anytime! Let me know if you need anything else."],
        'score_threshold': 1
    },

    # Intent 5: Product Inquiry (requires two core keywords for a high score)
    'product_query': {
        'keywords': ['product', 'service', 'buy', 'offer', 'price', 'cost'],
        'responses': ["Could you tell me the name of the product you are interested in?", "Our main product is the 'Simple AI' software. Would you like to know the price?"],
        'score_threshold': 2
    }
}

print("Expanded Intents and Keywords defined successfully.")

Expanded Intents and Keywords defined successfully.


In [10]:
def preprocess_input(text):
    """
    Checklist: Preprocess user input (lowercase, remove punctuation, tokenization/stemming).
    """
    # 1. Lowercase and remove punctuation
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)

    # 2. Tokenization and Stemming
    tokens = nltk.word_tokenize(text)
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    return stemmed_tokens

def match_intent(tokens):
    """
    Checklist: Match input to intents using keywords.
    Finds the intent with the highest keyword count.
    """
    best_match = {'intent': 'fallback', 'score': 0}

    for intent_name, data in intents.items():
        score = 0

        # Count keyword matches against stemmed user input
        for keyword in data['keywords']:
            stemmed_keyword = stemmer.stem(keyword)
            if stemmed_keyword in tokens:
                score += 1

        # This intent is a better match if it has a higher score
        if score > best_match['score']:
            best_match['intent'] = intent_name
            best_match['score'] = score

    # Final check: Does the best match meet its required threshold?
    # This prevents a low score (e.g., 1) from matching a complex intent (e.g., 'product_query')
    if best_match['intent'] != 'fallback' and best_match['score'] >= intents[best_match['intent']]['score_threshold']:
        return best_match['intent']
    else:
        # Fallback if no intent meets its specific score requirement
        return 'fallback'

print("Enhanced Preprocessing and Matching functions defined.")

Enhanced Preprocessing and Matching functions defined.


In [11]:
# Assuming all functions and the 'intents' dictionary from previous cells are defined

def generate_response(intent_name):
    # (Function body remains the same as previous response, defining fallback and response choice)

    fallback_responses = [
        "I'm still learning and don't have an answer for that yet. Could you try asking about hours or our products?",
        "Sorry, I didn't catch that. Please try rephrasing your question.",
        "That's outside my current capabilities. How about a topic I know, like our business hours?",
        "I'm not sure how to respond. Type 'help' to see what I can answer."
    ]

    if intent_name in intents:
        return random.choice(intents[intent_name]['responses'])
    else:
        return random.choice(fallback_responses)


def display_help():
    print("\n--- I can answer questions about the following topics: ---")
    for name in intents.keys():
        if name != 'thanks': # Don't list 'thanks' as a primary topic
            print(f"- **{name.replace('_', ' ').title()}** (e.g., 'What are your {name.replace('get_', '')}?', 'Tell me about your {name.replace('_query', '')}s.')")
    print("\nTo stop chatting, type 'quit' or 'exit'.")
    print("-------------------------------------------------------------")

def chatbot_loop():
    print("\n-------------------------------------------------------------")
    print("      Simple Intent Chatbot (Type 'quit' or 'exit' to end)     ")
    display_help()

    # Loop for continuous conversation
    while True:
        user_input = input("You: ")

        # Check for exit command
        if user_input.lower() in ['quit', 'exit']:
            print("\nBot: Goodbye! Always here to chat. Come back soon!")
            break

        # Check for help command
        if user_input.lower() == 'help':
            display_help()
            continue

        # 1. Preprocess the input
        processed_tokens = preprocess_input(user_input)

        # 2. Match the intent
        intent_match = match_intent(processed_tokens)

        # 3. Generate and print the response
        response = generate_response(intent_match)
        print(f"Bot: {response}")

# Start the chat loop
chatbot_loop()


-------------------------------------------------------------
      Simple Intent Chatbot (Type 'quit' or 'exit' to end)     

--- I can answer questions about the following topics: ---
- **Greeting** (e.g., 'What are your greeting?', 'Tell me about your greetings.')
- **Get Hours** (e.g., 'What are your hours?', 'Tell me about your get_hourss.')
- **Bot Name** (e.g., 'What are your bot_name?', 'Tell me about your bot_names.')
- **Product Query** (e.g., 'What are your product_query?', 'Tell me about your products.')

To stop chatting, type 'quit' or 'exit'.
-------------------------------------------------------------
You: quit

Bot: Goodbye! Always here to chat. Come back soon!


In [12]:
# Checklist: Save chatbot script/notebook. (Download your Colab file for safety.)

print("\n--- Project Documentation and Summary ---")

# Checklist: Document intents, preprocessing choices, and limitations.
print("\n**1. Intents Defined and Functionality:**")
for name, data in intents.items():
    keywords = ', '.join(data['keywords'])
    print(f" - {name.capitalize()}: Matches keywords like '{keywords.split(', ')[0]}' and requires {data['score_threshold']} keyword match(es).")

print("\n**2. Preprocessing Choices:**")
print(" - Logic: Custom Python function using NLTK.")
print(" - Techniques: Lowercasing, Punctuation Removal, Tokenization, and **Stemming** (to group words like 'hours' and 'hourly').")
print(" - Matching: Intent is classified based on the *highest count* of matching keywords that also meets a minimum threshold.")

print("\n**3. Limitations and Future Improvements:**")
print(" - Lack of State/Context: The bot treats every message as the first one.")
print(" - Rigid Matching: Complex sentences where keywords are diluted may lead to the fallback response.")
print(" - No Learning: The bot's logic is hardcoded and does not improve from conversations.")

print("\nProject complete! Save your Colab notebook. You successfully created a fundamental Intent-Based Chatbot.")


--- Project Documentation and Summary ---

**1. Intents Defined and Functionality:**
 - Greeting: Matches keywords like 'hello' and requires 1 keyword match(es).
 - Get_hours: Matches keywords like 'open' and requires 1 keyword match(es).
 - Bot_name: Matches keywords like 'name' and requires 2 keyword match(es).
 - Thanks: Matches keywords like 'thank' and requires 1 keyword match(es).
 - Product_query: Matches keywords like 'product' and requires 2 keyword match(es).

**2. Preprocessing Choices:**
 - Logic: Custom Python function using NLTK.
 - Techniques: Lowercasing, Punctuation Removal, Tokenization, and **Stemming** (to group words like 'hours' and 'hourly').
 - Matching: Intent is classified based on the *highest count* of matching keywords that also meets a minimum threshold.

**3. Limitations and Future Improvements:**
 - Lack of State/Context: The bot treats every message as the first one.
 - Rigid Matching: Complex sentences where keywords are diluted may lead to the fallba